In [41]:
from pathlib import Path
import tidytcells as tt
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent.parent
COMBINED_CLEAN = ROOT / "processed" / "combined_trb_clean.csv"

combined_clean = pd.read_csv(COMBINED_CLEAN, low_memory=False)

In [42]:
combined_clean.columns

Index(['cdr3', 'tcr_chain', 'v_gene', 'j_gene', 'peptide', 'mhc_a', 'mhc_b',
       'mhc_class', 'source'],
      dtype='str')

In [43]:
combined_clean['j_gene'].unique()

<StringArray>
['TRBJ2-1*01', 'TRBJ2-2*01', 'TRBJ2-3*01', 'TRBJ2-7*01', 'TRBJ1-1*01',
 'TRBJ1-5*01', 'TRBJ2-5*01', 'TRBJ1-2*01', 'TRBJ1-6*01', 'TRBJ1-3*01',
 'TRBJ2-4*01', 'TRBJ1-4*01', 'TRBJ2-6*01', 'TRBJ1-6*02',    'TRBJ2-3',
    'TRBJ2-1',    'TRBJ1-2',    'TRBJ1-4',    'TRBJ2-7',    'TRBJ1-1',
    'TRBJ1-5',    'TRBJ2-2',    'TRBJ1-6',    'TRBJ1-3',    'TRBJ2-5',
    'TRBJ2-6',    'TRBJ2-4',     'TRAJ30',     'TRAJ26', 'TRBJ2-7*02']
Length: 30, dtype: str

In [44]:
help(tt.tr)

Help on package tidytcells.tr in tidytcells:

NAME
    tidytcells.tr - Functions to manage TR gene data.

PACKAGE CONTENTS
    _get_aa_sequence
    _query
    _standardize

FILE
    c:\users\chemegrad2025\desktop\peptcr\tcr_peptide_interactions\.venv\lib\site-packages\tidytcells\tr\__init__.py




In [45]:
help(tt.tr.standardize)

Help on function standardize in module tidytcells.tr._standardize:

standardize(symbol: Optional[str] = None, species: Optional[str] = None, enforce_functional: Optional[bool] = None, precision: Optional[Literal['allele', 'gene']] = None, on_fail: Optional[Literal['reject', 'keep']] = None, log_failures: Optional[str] = None, gene: Optional[str] = None, suppress_warnings: Optional[bool] = None) -> Optional[str]
    Attempt to standardize a TR gene / allele symbol to be IMGT-compliant.

    .. topic:: Supported species

        - ``"homosapiens"``
        - ``"musmusculus"``

    :param symbol:
        Potentially non-standardized TR gene / allele symbol.
    :type symbol:
        str
    :param species:
        Can be specified to standardise to a TR symbol that is known to be valid for that species (see above for supported species).
        If set to ``"any"``, then first attempts standardisation for *Homo sapiens*, then *Mus musculus*.
        Defaults to ``"homosapiens"``.

        

In [ ]:
import torch
import torch.utils._config_module as _cm
import numpy as np
from sklearn.decomposition import PCA
from transformers import AutoModelForMaskedLM

# torch._dynamo.config.recompile_limit was added in PyTorch 2.5.
# Patch ConfigModule.__setattr__ to tolerate unknown attributes during import.
_orig_setattr = _cm.ConfigModule.__setattr__
def _lenient_setattr(self, name, value):
    try:
        _orig_setattr(self, name, value)
    except AttributeError:
        object.__setattr__(self, name, value)
_cm.ConfigModule.__setattr__ = _lenient_setattr

model = AutoModelForMaskedLM.from_pretrained("Synthyra/ESMplusplus_large", trust_remote_code=True)

_cm.ConfigModule.__setattr__ = _orig_setattr  # restore strict validation

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
tokenizer = model.tokenizer

print(type(model).__name__, "loaded on", device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

In [47]:
example_data = combined_clean.iloc[::1000].copy()

In [48]:
import stitchr_utils as su

vectorized_stitch = np.vectorize(su.get_full_aa_sequence)

# Pass the columns directly!
example_data["full_aa_sequence"] = vectorized_stitch(
    example_data["v_gene"], 
    example_data["j_gene"], 
    example_data["cdr3"], 
    "HUMAN", 
    True # no_leader flag
)

example_data.tail(50)

,cdr3,tcr_chain,v_gene,j_gene,peptide,mhc_a,mhc_b,mhc_class,source,full_aa_sequence
32000,CACYLGQILPGEQYF,TRB,TRBV7-2*01,TRBJ2-7*01,GLCTLVAML,HLA-A*02:01,NaN,MHCI,IEDB,GAGVSQSPSNKVTEKGKDVELRCDPISGHTALYWYRQSLGQGLEFL...
33000,CARRLAGGGPSGYTF,TRB,TRBV5-1*01,TRBJ1-2*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,KAGVTQTPRYLIKTRGQQVTLSCSPISGHRSVSWYQQTPGQGLQFL...
34000,CASIQSGSVYEQYF,TRB,TRBV7-2*01,TRBJ2-7*01,GLCTLVAML,HLA-A*02:01,NaN,MHCI,IEDB,GAGVSQSPSNKVTEKGKDVELRCDPISGHTALYWYRQSLGQGLEFL...
35000,CASSDPRDRVGETQYF,TRB,TRBV7-3*01,TRBJ2-5*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,GAGVSQTPSNKVTEKGKYVELRCDPISGHTALYWYRQSLGQGPEFL...
36000,CASSFGLAGPLNEQFF,TRB,TRBV7-7*01,TRBJ2-1*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,GAGVSQSPRYKVTKRGQDVTLRCDPISSHATLYWYQQALGQGPEFL...
37000,CASRQLLGSNQPQRF,TRB,TRBV6-5*01,TRBJ1-5*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,NAGVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLI...
38000,CASSALLGSASTLYF,TRB,TRBV6-1*01,TRBJ2-3*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,NAGVTQTPKFQVLKTGQSMTLQCAQDMNHNSMYWYRQDPGMGLRLI...
39000,CASSLGQGEHQPQHF,TRB,TRBV7-7*01,TRBJ1-5*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,GAGVSQSPRYKVTKRGQDVTLRCDPISSHATLYWYQQALGQGPEFL...
40000,CASSLDRRETRYF,TRB,TRBV12-1,TRBJ2-5*01,GLCTLVAML,HLA-A*02:01,NaN,MHCI,IEDB,DAGVIQSPRHKVTEMGQSVTLRCEPISGHNDLLWYRQTFVQGLELL...
41000,CASSLAGGGTETQYF,TRB,TRBV6-5*01,TRBJ2-5*01,YVLDHLIVV,HLA-A*02:01,NaN,MHCI,IEDB,NAGVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLI...
